# Stockout-Aware Forecasting for Perishable Goods: Reproduction Notebook

Reproduces all results of the paper in Google Colab. Run the cells in order.

All analysis code is in `src/`; this notebook only calls those scripts. Random seed: 42 (`src/config.py`).
Total run time on a free Colab CPU runtime: about 80 minutes (the forecasting step takes about 65).

## 1. Set up the project folder

**Option A – Google Drive** (data and results persist between sessions): the folder is found automatically.

**Option B – GitHub clone** (temporary session): comment out Option A and uncomment Option B.

In [ ]:
import glob, os

# Option A: Google Drive. The project folder is found automatically anywhere in
# My Drive (the folder that contains src/config.py).
from google.colab import drive
drive.mount('/content/drive')
found = glob.glob('/content/drive/MyDrive/**/perishable-forecast/src/config.py', recursive=True)
newest = lambda f: max(os.path.getmtime(p) for p in glob.glob(os.path.join(os.path.dirname(f), '*.py')))
candidates = sorted(found, key=newest, reverse=True)   # most recently updated copy first
PROJECT_DIR = os.path.dirname(os.path.dirname(candidates[0])) if candidates else None

# Option B: clone from GitHub (comment out Option A above)
# !git clone https://github.com/Dr-Archana-Sasi/perishable-forecast.git /content/perishable-forecast
# PROJECT_DIR = '/content/perishable-forecast'

if PROJECT_DIR is None:
    raise FileNotFoundError('Project folder not found in My Drive; set PROJECT_DIR manually.')
if len(candidates) > 1:
    print('Several copies found; using the most recently updated one. Others:',
          [os.path.dirname(os.path.dirname(c)) for c in candidates[1:]])
os.chdir(PROJECT_DIR)
print('Working directory:', os.getcwd())

## 2. Install dependencies
Run at the start of every session.

In [ ]:
!pip install -q -r requirements.txt

## 3. Download the dataset (FreshRetailNet-50K, CC BY 4.0)
Needed only once when using Google Drive.

In [ ]:
!python src/download_data.py

## 4. Exploratory data analysis
Tables T1–T4, Figures F1–F5.

In [ ]:
!python src/eda.py

## 5. Demand recovery
Recovered daily demand, validation by synthetic censoring. Tables T5–T7, Figures F6–F7.

In [ ]:
!python src/recover_demand.py

## 6. Forecasting
Seasonal naive, Empirical-28, ETS and LightGBM, each trained on recorded sales and on recovered demand. Tables T8–T10b, Figure F8. About 65 minutes.

In [ ]:
!python src/forecast.py

## 7. Inventory simulation
Fill rate, waste and cost for each forecast set. Tables T11–T14, Figures F9–F10.

In [ ]:
!python src/inventory_sim.py

## 8. Robustness of demand recovery
Alternative recovery methods and parameter sensitivity. Tables T15–T16, Figure F11. About 2 minutes.

In [ ]:
!python src/recovery_robustness.py

## 9. Rolling-origin backtest
Three additional test weeks. Tables T17–T19, Figure F12. About 2.5 hours; if the session disconnects, run the cell again and it continues where it stopped.

In [ ]:
!python src/backtest.py

## 10. Record software versions
Writes `results/environment.txt`.

In [ ]:
!python src/record_environment.py

## 11. View the figures

In [ ]:
from IPython.display import display, Image
import glob
for f in sorted(glob.glob('results/figures/*.png'), key=lambda p: int(p.split('/F')[-1].split('_')[0])):
    print(f)
    display(Image(f, width=800))